
# 04 · 闭环状态估计：滤波变好了吗？

本课把第 03 课的测量边界接进真实 MetaDrive 闭环。`run_episode` 仍把 `before_*` 保存为 truth，但 planner 与 controller 只能看到 `input_*`。比较 oracle、raw measurement、causal scalar Kalman 三组：同时报告测量 RMSE、横向误差、距离、终止原因和动作轨迹。

滤波使用一维 random-walk Kalman：`P⁻=P+Q·dt`，`K=P⁻/(P⁻+R)`，`x=x⁻+K(z-x⁻)`。它假定一个决策间隔内状态近似平稳，用带单位的 Q 留出运动余量。它是教学模型，不能直接当作车辆定位方案。


In [ ]:

from dataclasses import replace
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src" / "ad_tutorial").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from ad_tutorial.driving import DrivingObservation, DrivingConfig, run_episode
from ad_tutorial.estimation import (
    KalmanObserver, MeasurementConfig, RawMeasurementObserver, ScalarKalman,
    ego_to_world, make_observer, world_to_ego,
)



## 1. 两步手算：R、Q·dt、后验 P

例如 x 测量的 `σ=0.18m`，所以 `R=σ²=0.0324m²`；`Q=0.01m²/s`，`dt=0.1s`，一次预测增加 `Q·dt=0.001m²`。第一笔样本初始化状态，第二笔样本执行 `P⁻=P+Q·dt`，再算 `K`、后验状态和 `P=(1-K)P⁻`。每个物理量使用自己单位的 Q/R。


In [ ]:

R = 0.18 ** 2
Q_per_s = 0.01
dt = 0.1
kf = ScalarKalman(R, process_variance=Q_per_s)
x1 = kf.update(10.20, dt)
P1 = kf.variance
P_minus = P1 + Q_per_s * dt
K = P_minus / (P_minus + R)
x2 = kf.update(10.00, dt)
print({"R_m2": R, "Qdt_m2": Q_per_s * dt, "P_minus_m2": P_minus,
       "K": K, "x_after_two_m": x2, "P_posterior_m2": kf.variance,
       "P_formula_m2": (1 - K) * P_minus})
assert np.isclose(kf.variance, (1 - K) * P_minus)


In [ ]:

base = DrivingConfig(seed=7, horizon=120)
sensor = MeasurementConfig(seed=19)  # same noise, bias and Q as the default CLI
results = {mode: run_episode(base, observer=make_observer(mode, sensor))
           for mode in ("oracle", "raw", "filter")}

def rmse(result, estimate, truth):
    return float(np.sqrt(np.mean([(r[estimate] - r[truth]) ** 2 for r in result.trace])))

def signed_error(result, estimate, truth):
    return float(np.mean([r[estimate] - r[truth] for r in result.trace]))

for mode, result in results.items():
    print(mode, {
        "measurement_x_rmse_m": rmse(result, "measurement_x_m", "before_x_m"),
        "input_x_rmse_m": rmse(result, "input_x_m", "before_x_m"),
        "input_x_signed_error_m": signed_error(result, "input_x_m", "before_x_m"),
        "input_lateral_rmse_m": rmse(result, "input_lateral_error_m", "before_lateral_error_m"),
        "mean_abs_lateral_error_m": result.metrics["mean_abs_lateral_error_m"],
        "distance_m": result.metrics["distance_traveled_m"],
        "outcome": result.metrics["outcome"],
    })
print("required observation: default filter failure=", results["filter"].metrics["failure"],
      "filter outcome=", results["filter"].metrics["outcome"],
      "; random-walk filter lags moving x; Q tuning is a trade-off, not a repair")



## 1. 看两条链：估计误差与真实驾驶结果

上图的真值曲线只用于评估，不能进入控制器；`measurement_*` 是 filter 看到的当前原始测量，`input_*` 才是控制器输入。默认 120 步配置中，filter 的 x-position RMSE 会高于原始 measurement，并在本固定场景提前失败；这正是缺少运动模型的反例。若 filter RMSE 下降但横向误差或距离变差，这是可以发生的：平滑带来滞后，噪声变小不等于闭环一定更好。


In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(10, 9), constrained_layout=True)
colors = {"oracle": "#222222", "raw": "#246b89", "filter": "#c74c3c"}
for mode, result in results.items():
    t = [r["time_s"] - result.metrics["decision_dt_s"] for r in result.trace]
    color = colors[mode]
    axes[0].plot(t, [r["before_longitudinal_m"] for r in result.trace], "--", color=color, alpha=.6, label=f"{mode} truth")
    axes[0].plot(t, [r["measurement_longitudinal_m"] for r in result.trace], ":", color=color, alpha=.8, label=f"{mode} measurement")
    axes[0].plot(t, [r["input_longitudinal_m"] for r in result.trace], color=color, label=f"{mode} estimate")
    axes[1].plot(t, [r["input_x_m"] - r["before_x_m"] for r in result.trace], color=color, label=mode)
    axes[2].plot(t, [r["command_steering"] for r in result.trace], color=color, label=mode)
axes[0].set(xlabel="Time / s", ylabel="longitudinal s / m", title="Truth, measurement and estimate")
axes[1].set(xlabel="Time / s", ylabel="signed x error / m", title="Estimate minus truth")
axes[2].set(xlabel="Time / s", ylabel="steering command")
for ax in axes:
    ax.grid(alpha=.2); ax.legend()
plt.show()



## 2. 因果性与同噪声检查

观测器接口是 `observe(truth, step, dt)`。闭环会在 reset 后提供固定 lane，使 noisy world pose 能投影成 `s/e_y`；它只收到当前行 truth，没有历史轨迹参数，也没有 future truth。filter 的内部状态只由过去的 measurement 更新。下面的 probe 记录每次调用，验证调用顺序、`reset()` 次数和控制输入确实变化。


In [ ]:

class ProbeObserver(RawMeasurementObserver):
    def __init__(self, config):
        self.calls = []
        self.reset_calls = 0
        super().__init__(config)
    def reset(self, lane=None):
        self.reset_calls += 1
        super().reset(lane)
    def observe(self, truth, step, dt):
        self.calls.append((step, dt, truth.lane_lateral_m))
        return super().observe(truth, step, dt)

probe = ProbeObserver(sensor)
probe_result = run_episode(DrivingConfig(seed=7, horizon=8), observer=probe)
assert probe.reset_calls == 2  # constructor + exactly one run reset
assert [call[0] for call in probe.calls] == list(range(len(probe_result.trace)))
assert not np.allclose([r["input_lateral_error_m"] for r in probe_result.trace],
                       [r["before_lateral_error_m"] for r in probe_result.trace])
print("reset calls:", probe.reset_calls, "observe calls:", len(probe.calls))



## 3. 可编辑练习：不要追求单一赢家

1. 固定 `seed=19`，把位置噪声和 lateral 噪声分别放大 2 倍；预测 raw/filter 的 RMSE 和驾驶指标如何变化。
2. 固定噪声，把有 m²/s 单位的 `position_process_variance_m2_per_s` 改成 `0.0` 与 `0.10`。哪个更容易滞后？用 input 曲线与 command 曲线解释；朝向和速度的 Q 使用各自带单位字段。
3. 让初始偏移变成 `-0.4m`，不调增益，比较三组的失败原因。

参考答案：更大的 R 通常让 raw 更抖，filter 可能更依赖历史；更大的 Q 提高跟随新测量的速度、减少滞后，但会放进更多噪声。没有哪组在所有指标上必胜；必须保留相同 seed、噪声配置和完整 trace，分别报告 RMSE 与真实驾驶结果。短时跑满 horizon 也不等于到达终点。


In [ ]:

changed = replace(sensor, position_process_variance_m2_per_s=0.10)
fast_filter = run_episode(base, observer=KalmanObserver(changed))
print("Q=0.10 filter:", fast_filter.metrics["mean_abs_lateral_error_m"],
      "lateral RMSE:", rmse(fast_filter, "input_lateral_error_m", "before_lateral_error_m"))



## 4. 从课程机制到研究问题

本单元只做仿真机制实验：真值用于合成测量并评估，固定单车道与几何控制器。下一步可研究速度/转弯模型、异步传感器、观测丢失、真实地图标定，再考虑学习型 perception。不要因为一条滤波曲线更平滑就声称定位更可靠；先问状态、坐标、延迟、协方差和失效证据是否匹配。

CLI 会把 `oracle_seed*.json/csv/png`、`raw_*`、`filter_*`、`summary.json` 和比较图保存到 `artifacts/state_estimation/`，便于从轨迹重算指标。
